In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import CoolProp.CoolProp as CP
import thermo as thermo
import time as timer
import cProfile
import pstats
import yaml
import copy
import sys

In [3]:
def gr_emis(temp: float) -> float:
    """
    Calculate the emissivity of graphite as a function of temperature.
    
    Parameters:
    temp (float): Temperature (K).
    
    Returns:
    float: Emissivity of graphite.
    """
    return 0.679 + 6e-5 * temp

def storage_cp(temp):
    #storage graphite specific heat
    #replaced with fixed cp until better enthalpy handling implemented
    cp_arr= np.array([1.27, 1.92, 1.99, 2.13])
    temp_C = np.array([300, 750, 1200, 1995])
    temp_C = np.append(temp_C,2400)
    cp_arr = np.append(cp_arr, 2.13)
    temp_C_interp = temp - 273.15
    # Extrapolate up to 2400°C
    if temp_C_interp > 2400:
        temp_C_interp = 2400
    return 2000   
    #return np.interp(temp_C_interp, temp_C, cp_arr) * 1000

def storage_tcx(temp):
    #storage graphite conductivity
    temp_C = np.array([300, 750, 1200, 1995])
    conductivity = np.array([31.5, 28.7, 24.8, 23.1])
    temp_C_interp = temp - 273.15
    temp_C = np.append(temp_C,2400)
    conductivity = np.append(conductivity, conductivity[-1] + (conductivity[-1] - conductivity[-2]) / (temp_C[-2] - temp_C[-3]) * (temp_C[-1] - temp_C[-2]))
        # Extrapolate up to 2400°C
    if temp_C_interp > 2400:
        temp_C_interp = 2400
    return 15
    #return np.interp(temp_C_interp, temp_C, conductivity)

def ins_tcx(temp):
    #Carbon insulation thermal conductivity in Argon
    temps = np.array([0, 400, 800, 1000, 2000, 2500])
    tcx = np.array([0.09, 0.09, 0.176, 0.35, 0.94, 0.94 + (0.94 - 0.35) * 500 / 1000]) #data for 1000 and 2000 C from manufacturer. Linear extrapolation to 2500 C, flat down to 0C
    temp_C = temp - 273.15
    return np.interp(temp_C, temps, tcx)

def wall_tcx(temp):
    return 0.018  #SES wall thermal conductivity in W/mK